# Forecast Evaluation Review

This review presents the residential water-demand forecasting results. It uses persisted forecasts and diagnostics, preserves the chronological evaluation design, and separates synthetic benchmark evidence from conclusions that require real observations.

## Evaluation protocol

The accepted windows contain 108 training months, 24 validation months, and 36 untouched test months. No model is retrained or retuned in this review. All tables and figures are reconstructed from persisted forecasts, diagnostics, and chronological demand records.

In [19]:
from pathlib import Path
import importlib
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.evaluation.phase6_evaluation as phase6_evaluation
phase6_evaluation = importlib.reload(phase6_evaluation)

from src.evaluation.phase6_evaluation import (
    build_phase6_baseline_comparison,
    build_phase6_horizon_review,
    build_phase6_regime_analysis,
    build_phase6_regime_error_summary,
    build_phase6_residual_analysis,
    build_phase6_scorecard,
    build_phase6_selection_review,
    validate_phase5_inputs,
)

inputs = validate_phase5_inputs(PROJECT_ROOT)
scorecard = build_phase6_scorecard(PROJECT_ROOT)
analysis = build_phase6_residual_analysis(PROJECT_ROOT)
selection = build_phase6_selection_review(PROJECT_ROOT)
baseline = build_phase6_baseline_comparison(PROJECT_ROOT)
horizon_review = build_phase6_horizon_review(PROJECT_ROOT)
regimes = build_phase6_regime_analysis(PROJECT_ROOT)
regime_errors = build_phase6_regime_error_summary(PROJECT_ROOT)
print(f"Validation months: {len(inputs['artifacts']['validation_y'])}")
print(f"Test months: {len(inputs['artifacts']['test_y'])}")
print(f"Demand regimes: {regimes['selected_k']}")

Validation months: 24
Test months: 36
Demand regimes: 2


## Model performance

In [20]:
display(scorecard.round(3))
display(selection.round(3))

,model,split,mae,rmse,mape,smape,r2
0,ANN,test,375856.590,450990.485,9.510,9.013,0.609
1,LSTM,test,404871.042,497991.624,9.541,9.696,0.523
2,Seasonal naive,test,433726.936,526201.042,9.915,10.559,0.467
3,Random Forest,test,486348.093,584281.741,11.454,11.838,0.343
4,ANN,validation,227344.522,284796.500,5.921,5.770,0.797
5,LSTM,validation,255569.662,289081.453,6.548,6.558,0.791
6,Random Forest,validation,286119.370,345121.157,7.105,7.345,0.702
7,Seasonal naive,validation,291879.192,357628.720,7.278,7.533,0.680


,split,rmse_leader,rmse,mae_leader,mae,rmse_leader_matches_mae_leader,test_rmse_leader_matches_validation
0,test,ANN,450990.485,ANN,375856.590,True,True
1,validation,ANN,284796.500,ANN,227344.522,True,True


In [21]:
scorecard_plot = scorecard.pivot(index='model', columns='split', values='rmse')
scorecard_plot.plot(kind='bar', figsize=(10, 5), color=['#d9480f', '#2f6f9f'], title='RMSE across evaluation windows')
plt.ylabel('RMSE')
plt.xlabel('')
plt.tight_layout()
plt.show()

C:\Users\derai\AppData\Local\Temp\ipykernel_16376\11777696.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comparison with the seasonal benchmark

Negative differences indicate lower error than the seasonal-naive benchmark. These comparisons describe the current synthetic benchmark only.

In [22]:
display(baseline.round(3))

,model,split,baseline_mae,model_mae,mae_delta_vs_baseline,mae_relative_change_pct,baseline_rmse,model_rmse,rmse_delta_vs_baseline,rmse_relative_change_pct
0,ANN,test,433726.936,375856.590,-57870.346,-13.343,526201.042,450990.485,-75210.557,-14.293
1,LSTM,test,433726.936,404871.042,-28855.894,-6.653,526201.042,497991.624,-28209.418,-5.361
2,Random Forest,test,433726.936,486348.093,52621.157,12.132,526201.042,584281.741,58080.699,11.038
3,ANN,validation,291879.192,227344.522,-64534.669,-22.110,357628.720,284796.500,-72832.221,-20.365
4,LSTM,validation,291879.192,255569.662,-36309.529,-12.440,357628.720,289081.453,-68547.267,-19.167
5,Random Forest,validation,291879.192,286119.370,-5759.821,-1.973,357628.720,345121.157,-12507.564,-3.497


## Residual behavior and uncertainty

Residual summaries describe bias, spread, typical error, extreme error, and central quantiles. They are descriptive uncertainty summaries rather than operational confidence intervals.

In [23]:
display(analysis['diagnostics'].round(3))
display(analysis['stability'].round(3))
display(analysis['large_errors'].head(12))

,model,split,mean_residual,residual_std,residual_mae,residual_rmse,max_absolute_residual,p05_residual,p95_residual
0,Seasonal naive,validation,175864.467,311400.050,291879.192,357628.720,752276.000,-293589.200,583251.315
1,Seasonal naive,test,368900.492,375233.213,433726.936,526201.042,1276016.700,-239827.575,1026870.675
2,Random Forest,validation,167325.614,301845.576,286119.370,345121.157,749287.124,-237242.683,624104.698
3,Random Forest,test,289689.490,507410.241,486348.093,584281.741,1239251.844,-742578.885,1018794.569
4,ANN,validation,-60404.231,278317.041,227344.522,284796.500,633124.806,-597950.782,284873.107
5,ANN,test,-175618.221,415392.174,375856.590,450990.485,917071.119,-907619.422,491489.944
6,LSTM,validation,61047.987,282561.905,255569.662,289081.453,568952.800,-374126.955,391098.245
7,LSTM,test,159729.881,471680.001,404871.042,497991.624,1102886.000,-584300.050,959144.200


,model,rmse_rank_test,rmse_rank_validation,mae_rank_test,mae_rank_validation,rmse_rank_change,mae_rank_change,ranking_stable
0,ANN,1.0,1.0,1.0,1.0,0.0,0.0,True
1,LSTM,2.0,2.0,2.0,2.0,0.0,0.0,True
2,Random Forest,4.0,3.0,4.0,3.0,1.0,1.0,False
3,Seasonal naive,3.0,4.0,3.0,4.0,-1.0,-1.0,False


,model,split,date,month,residual,absolute_error,error_rank
0,ANN,test,2022-06-01,6,-9.170711e+05,9.170711e+05,1
1,ANN,test,2024-02-01,2,-9.144127e+05,9.144127e+05,2
2,ANN,test,2023-08-01,8,-9.053550e+05,9.053550e+05,3
3,ANN,validation,2020-07-01,7,-6.331248e+05,6.331248e+05,1
4,ANN,validation,2021-04-01,4,-6.313847e+05,6.313847e+05,2
5,ANN,validation,2020-01-01,1,5.992606e+05,5.992606e+05,3
6,LSTM,test,2023-01-01,1,1.102886e+06,1.102886e+06,1
7,LSTM,test,2022-04-01,4,1.029366e+06,1.029366e+06,2
8,LSTM,test,2023-05-01,5,9.357368e+05,9.357368e+05,3
9,LSTM,validation,2020-01-01,1,5.689528e+05,5.689528e+05,1


In [24]:
seasonal_plot = analysis['seasonal_errors'].pivot_table(index='month', columns='model', values='mean_absolute_error')
seasonal_plot.plot(figsize=(12, 5), marker='o', title='Mean absolute error by calendar month')
plt.ylabel('Mean absolute error')
plt.xlabel('Calendar month')
plt.tight_layout()
plt.show()

C:\Users\derai\AppData\Local\Temp\ipykernel_16376\1390611536.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sequential forecast horizon

The horizon review compares the first, second, and third monthly forecasts from a common forecast origin. A quarterly interpretation remains the sum of these sequential monthly forecasts.

In [25]:
display(horizon_review.round(3))
horizon = inputs['horizon'].copy()
actual = horizon.drop_duplicates('horizon_step').sort_values('horizon_step')
actual.plot(x='horizon_step', y='actual_m3', marker='o', linewidth=2.5, figsize=(11, 5), label='Actual demand', title='Three-month sequential forecast review')
for model_name, frame in horizon.groupby('model'):
    ordered = frame.sort_values('horizon_step')
    plt.plot(ordered['horizon_step'], ordered['forecast_m3'], marker='o', label=model_name)
plt.xlabel('Months ahead')
plt.ylabel('Residential water demand (m3)')
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

,model,horizon_step,one_month_absolute_error,three_month_mae,three_month_rmse
0,ANN,1,343684.375,520400.426,520400.426
1,ANN,3,343684.375,520400.426,520400.426
2,LSTM,1,474509.500,841222.200,841222.200
3,LSTM,3,474509.500,841222.200,841222.200
4,Random Forest,1,604451.902,967632.846,967632.846
5,Random Forest,3,604451.902,967632.846,967632.846
6,Seasonal naive,1,93425.100,364010.500,364010.500
7,Seasonal naive,3,93425.100,364010.500,364010.500


C:\Users\derai\AppData\Local\Temp\ipykernel_16376\874593742.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Monthly demand regimes

The exploratory regime model uses training-period scaling and fitting, then assigns validation and test months without refitting. The clusters describe demand behavior rather than geography.

In [26]:
assignments = regimes['assignments'].copy()
assignments['date'] = pd.to_datetime(assignments['date'])
display(assignments.head())
display(regime_errors.round(3))
regime_sizes = assignments.groupby(['split', 'cluster']).size().unstack(fill_value=0)
regime_sizes.plot(kind='bar', stacked=True, figsize=(10, 5), title='Demand-regime membership by window')
plt.ylabel('Months')
plt.xlabel('')
plt.tight_layout()
plt.show()

,date,split,cluster
0,2022-01-01,test,0
1,2022-02-01,test,0
2,2022-03-01,test,0
3,2022-04-01,test,0
4,2022-05-01,test,0


,model,split,cluster,count,bias,mae,rmse
0,ANN,test,0,25,-136424.584,376671.640,447217.081
1,ANN,test,1,11,-264694.667,374004.202,459451.164
2,ANN,validation,0,16,-33329.008,220794.241,282244.584
3,ANN,validation,1,8,-114554.679,240445.085,289832.932
4,LSTM,test,0,25,265890.064,443552.576,540432.490
5,LSTM,test,1,11,-81543.264,316958.464,384490.426
6,LSTM,validation,0,16,119232.506,257489.869,291168.463
7,LSTM,validation,1,8,-55321.050,251729.250,284861.566
8,Random Forest,test,0,25,440255.313,535584.359,625296.978
9,Random Forest,test,1,11,-52505.561,374447.489,478155.334


C:\Users\derai\AppData\Local\Temp\ipykernel_16376\3036653846.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Reading the results

The scorecard, residual review, horizon comparison, and regime summaries provide a reproducible synthetic benchmark. They support model comparison and error interpretation, but they do not establish operational accuracy, causal relationships, or real-world deployment readiness. Those questions must be revisited after the real-data replacement and data-quality review.